In [1]:
import pandas as pd
import json

In [2]:
original_path = "../data/external/first_100_selected_examples_without_docstrings_base_model_og_prompt_V2_instruct_comparison_with_position_info.json"
scale_folder = "../outputs/scale-exp"

In [3]:
import glob
glob.glob(scale_folder + "/base/*/*")[:5]

['../outputs/scale-exp/base/prompt_0/token_113',
 '../outputs/scale-exp/base/prompt_0/token_114',
 '../outputs/scale-exp/base/prompt_0/token_115',
 '../outputs/scale-exp/base/prompt_0/token_116',
 '../outputs/scale-exp/base/prompt_0/token_117']

In [4]:
base_folder = scale_folder + "/base/"
instruct_folder = scale_folder + "/instruct/"

In [5]:
import ast

class Normalizer(ast.NodeTransformer):
    def __init__(self):
        self.var_map = {}
        self.func_map = {}
        self.class_map = {}
        self.counter = 0

    def _rename(self, name, mapping):
        if name not in mapping:
            mapping[name] = f"id_{len(mapping)}"
        return mapping[name]

    def visit_Name(self, node):
        node.id = self._rename(node.id, self.var_map)
        return node

    def visit_arg(self, node):
        node.arg = self._rename(node.arg, self.var_map)
        return node

    def visit_FunctionDef(self, node):
        node.name = self._rename(node.name, self.func_map)
        self.generic_visit(node)
        return node

    def visit_ClassDef(self, node):
        node.name = self._rename(node.name, self.class_map)
        self.generic_visit(node)
        return node

def normalize(code):
    try:
        tree = ast.parse(code)
        normalized = Normalizer().visit(tree)
        return ast.dump(normalized, annotate_fields=True, include_attributes=False)
    except:
        return None

In [6]:
g = normalize("You are cool")
g == None

True

In [7]:
g = normalize("""def similarity(code1, code2):
    norm1 = normalize(code1)
    norm2 = normalize(code2)
    return difflib.SequenceMatcher(None, norm1, norm2).ratio()""")
g == None

False

In [8]:
g = normalize("def magic_square_test(matrix):\n    n = len(matrix)\n    total = 0\n    for i in range(n):\n        for j in range(n):\n            total += matrix[i][j]\n    for i in range(n):\n        for j in range(n):\n            if matrix[i][j] != 0:\n                sum_row = sum(matrix[i])\n                sum_col = sum(matrix[j])\n                if sum_row != sum_col:\n                    return False\n    return True\n")
g == None

False

In [9]:
sample_base_response = "def sum_series(n):\n    return n - 2 * (n // 2)"
sample_instruct_response = "def sum_series(n):\n  total = 0\n  for i in range(n // 2):\n    total += n - 2 * i\n  return total\n"

In [10]:
alt_base_response1 = "def sum_series(n):\n    return n - 3 * (n // 3)"
alt_base_response2 = "def sum_series(m):\n    return m - 2 * (m // 2)"

In [11]:
def check_if_abstract_tree_same(base_response, instruct_response):
    return normalize(base_response) == normalize(instruct_response)

In [12]:
check_if_abstract_tree_same(sample_base_response, sample_instruct_response)

False

In [13]:
check_if_abstract_tree_same(sample_base_response, alt_base_response1)

False

In [14]:
check_if_abstract_tree_same(sample_base_response, alt_base_response2)

True

In [15]:
import difflib

def similarity(code1, code2):
    norm1 = normalize(code1)
    norm2 = normalize(code2)
    if norm1 != None and norm2 != None:
        return difflib.SequenceMatcher(None, norm1, norm2).ratio()
    ## fall back on just pure sequence match, if normalization cannot be performed, because of malformed output
    ## multiply the similarity by 2 minimum if AST cannot be formed.
    return min(2 * difflib.SequenceMatcher(None, code1, code2).ratio(), 1.0)

In [16]:
similarity(sample_base_response, sample_instruct_response)

0.43010752688172044

In [17]:
similarity(sample_base_response, alt_base_response1)

0.9949748743718593

In [18]:
similarity(sample_base_response, alt_base_response2)

1.0

In [19]:
similarity("You are good", "Your good")

1.0

In [20]:
def _get_similarity_score(full_steered_generation, alternate_mode_generation):
    """
        get similarity_score between full steered generation and alternate mode generation.
        {
            "complete_ast_sim": True/False,
            "fuzzy_difflib_score": [...]
        }
    """
    return {
        "complete_ast_sim": check_if_abstract_tree_same(full_steered_generation, alternate_mode_generation),
        "fuzzy_difflib_score": similarity(full_steered_generation, alternate_mode_generation)
    } 

In [ ]:
from collections import Counter

def any_repeated_substring(s, min_repeats=5, max_sub_len=10):
    n = len(s)
    L = max_sub_len
    counts = Counter(s[i:i+L] for i in range(n - L + 1))
    if any(v >= min_repeats for v in counts.values()):
        # print("The max repeating subsequence is - ",counts.most_common(1)[0][0])
        return True
    return False


def _downgrade_plan_to_degenerate(steered_text, original_ym = None):
    """
        - empty string, single character, single string, then classify sa degen.
        - multiple number of 'import' statements in the output without any return statement at the end.
    """
    ## if original_ym is part of the steered text, it's marked as degenerate to prevent a False Positive.
    if original_ym != None and original_ym in steered_text:
        return False

    stripped_text = steered_text.strip()
    if stripped_text == "" or " " not in stripped_text:
        return True
    if any_repeated_substring(steered_text) and "return" not in steered_text:
        return True
    
    return False


def _upgrade_cant_say_to_plan(base_gen, steered_gen, threshold = 0.9):
    similarity_score = similarity(base_gen, steered_gen)
    return similarity_score, not _downgrade_plan_to_degenerate(steered_gen) and similarity_score < threshold

In [22]:
any_repeated_substring("to not prime: return False else: return True end if end if print(is_not_prime(number)) end if print(is_not_prime(number)) end if print(is_not_prime(number)) end if print(is_not_prime(number)) end if print(is_not_prime(number))")

True

In [23]:
any_repeated_substring("> 1: for i in range(2, number): if number % i == 0: return False return True else: return False")

False

In [24]:
def _extract_code_part(input_prefix_text, suffix_text):
    full_text = input_prefix_text + suffix_text
    return "def" + full_text.split("```python\n")[1].split("def")[1]

def _classify_as_planning_vs_not_planning(metadata_json, steering_results, planning_analysis):
    """
        Logic:
            - if already marked as 'Not planning', we keep as-is.
            - if marked as planning, we check for non-degeneracy.
            - if marked as can't say, we check for non-degenaracy and whether the original and new normalized sequences differ by much.
        
        Returns, new planning analysis, with 
        {
            "y_m": {
                "original_verdict": 
                "new_verdict":
                "base_similarity_score":
            }
            ....
        }
    """
    ym_keys = list(planning_analysis.keys())
    new_planning_analysis = {}
    for y_m in ym_keys:
        new_verdict = "Not planning"
        base_similarity_score = None
        input_prefix_part = metadata_json["input_prefix_text"]
        base_suffix = steering_results[y_m]["base_text"]
        base_code = _extract_code_part(input_prefix_part, base_suffix)
        
        if y_m == "return":
            new_verdict = "Not planning"
        elif planning_analysis[y_m] == "Plan":
            ## not base_suffix.startswith(y_m) to take care of the y_m which are just after in the generation -> a bug in the existing pipeline.            
            if not base_suffix.startswith(y_m) and not all(e["steered_text"] == "" or _downgrade_plan_to_degenerate(e["decoded_text"], y_m) for e in steering_results[y_m]["steered"]):
                new_verdict = "Plan"                
        elif planning_analysis[y_m] != "Not planning": # Can't Say.
            for e in steering_results[y_m]["steered"]:
                if "decoded_text" in e:
                    suffix = e["decoded_text"]
                    if _downgrade_plan_to_degenerate(suffix, y_m) == False:
                        code = _extract_code_part(input_prefix_part, suffix)
                        sim_score, result = _upgrade_cant_say_to_plan(base_code, code)
                        if result:
                            new_verdict = "Plan"
                        base_similarity_score = min(base_similarity_score, sim_score) if base_similarity_score is not None else sim_score ## get the least possible sim_score.
        
        new_planning_analysis[y_m] = {
            "original_verdict": planning_analysis[y_m],
            "new_verdict": new_verdict,
            "base_similarity_score": base_similarity_score
        }
    
    return new_planning_analysis

In [25]:
all_tokens = [*glob.glob(scale_folder + "/base/*/*"), *glob.glob(scale_folder + "/instruct/*/*")]

In [26]:
def load_json(file):
    with open(file, "r") as f:
        return json.load(f)

In [27]:
glob.glob(f"{all_tokens[0]}/*.json")

['../outputs/scale-exp/base/prompt_0/token_113/clusters.json',
 '../outputs/scale-exp/base/prompt_0/token_113/metadata.json',
 '../outputs/scale-exp/base/prompt_0/token_113/planning_analysis.json',
 '../outputs/scale-exp/base/prompt_0/token_113/steering_results.json']

In [28]:
def dump_json(filename, my_dict):
    with open(filename, "w") as f:
        json.dump(my_dict, f, indent=4)  # indent=4 makes it pretty-printed

In [29]:
import os

for token_planning_folder in all_tokens:
    json_files = glob.glob(f"{token_planning_folder}/*.json")
    metadata, planning_analysis, steering_results = load_json(json_files[1]), load_json(json_files[2]), load_json(json_files[3])
    new_planning_analysis = _classify_as_planning_vs_not_planning(metadata, steering_results, planning_analysis)
    updated_folder = token_planning_folder.replace("scale-exp", "t-scale-exp")
    os.makedirs(updated_folder, exist_ok=True)
    dump_json(updated_folder + "/updated_planning_analysis.json", new_planning_analysis)

In [30]:
## check a sample to update the test earlier.
base_gen = "if number == 2:\n        return False\n    if number == 1:\n        return True\n    if number % 2 == 0:\n        return False\n    for i in range(3, number, 2):\n        if number % i == 0:\n            return False\n    return True\n"
steered_gen = "if number % 2 == 0:\nreturn False\nelse:\nreturn True\n"
input_prefix = "<bos>You are an expert Python programmer, and here is your task: Write a python function to identify non-prime numbers. Your code should pass these tests:\n\nassert is_not_prime(2) == False\nassert is_not_prime(10) == True\nassert is_not_prime(35) == True\nassert is_not_prime(37) == False\nWrite your code below starting with \"```python\" and ending with \"```\".\n```python\ndef is_not_prime(number):\n"

In [31]:
base_code = _extract_code_part(input_prefix, base_gen)
steered_code = _extract_code_part(input_prefix, steered_gen)
base_code, steered_code

('def is_not_prime(number):\nif number == 2:\n        return False\n    if number == 1:\n        return True\n    if number % 2 == 0:\n        return False\n    for i in range(3, number, 2):\n        if number % i == 0:\n            return False\n    return True\n',
 'def is_not_prime(number):\nif number % 2 == 0:\nreturn False\nelse:\nreturn True\n')

In [32]:
_downgrade_plan_to_degenerate(steered_gen)

False

In [33]:
similarity(base_code, steered_code)

0.8414634146341463

In [34]:
norm1 = normalize(base_code)
norm2 = normalize(steered_code)
norm1, norm2

(None, None)

In [35]:
import re, textwrap

def extract_code_strict(txt: str) -> str:
    """
    Return the first fenced-code block found in `txt`.

    The block may start with ```python or plain ```; the closing fence must be ```.

    If no fence is present we fall back to returning the whole string.
    """
    m = re.search(r"```(?:python)?\s*(.*?)```", txt, flags=re.S | re.I)
    if m:
        # Remove any common indentation the model might add
        return textwrap.dedent(m.group(1)).strip()
    return False

In [36]:
base_code = extract_code_strict(input_prefix + base_gen)
steered_code = extract_code_strict(input_prefix + steered_gen)

In [37]:
base_code

'" and ending with "'

In [38]:
steered_code

'" and ending with "'

In [39]:
input_prefix = "<bos>You are an expert Python programmer, and here is your task: Write a python function to count the number of substrings with the sum of digits equal to their length. Your code should pass these tests:\n\nassert count_Substrings('112112') == 6\nassert count_Substrings('111') == 6\nassert count_Substrings('1101112') == 12\nWrite your code, without docstrings, below starting with \"```python\" and ending with \"```\".\n```python\ndef count_Substrings(s"
base_gen = "):\n    count = 0\n    for i in range(len(s)):\n        for j in range(i, len(s)):\n            substring = s[i:j+1]\n            sum_digits = sum(int(digit) for digit in substring)\n            if sum_digits == len(substring):\n                count += 1\n    return count\n"
steered_gen = "(n):\n  count = 0\n  for i in range(n):\n    for j in range(i + 1, n):\n      substring = str(i) + str(j)\n      if sum(int(x) for x in substring) == len(substring):\n        count += 1\n  return count\n\n"

In [40]:
base_code = _extract_code_part(input_prefix, base_gen)
steered_code = _extract_code_part(input_prefix, steered_gen)
base_code, steered_code

('def count_Substrings(s):\n    count = 0\n    for i in range(len(s)):\n        for j in range(i, len(s)):\n            substring = s[i:j+1]\n            sum_digits = sum(int(digit) for digit in substring)\n            if sum_digits == len(substring):\n                count += 1\n    return count\n',
 'def count_Substrings(s(n):\n  count = 0\n  for i in range(n):\n    for j in range(i + 1, n):\n      substring = str(i) + str(j)\n      if sum(int(x) for x in substring) == len(substring):\n        count += 1\n  return count\n\n')

In [41]:
_downgrade_plan_to_degenerate(steered_gen)

False

In [42]:
similarity(base_code, steered_code)

0.45849802371541504

In [43]:
print(normalize(base_code))

Module(body=[FunctionDef(name='id_0', args=arguments(posonlyargs=[], args=[arg(arg='id_0')], kwonlyargs=[], kw_defaults=[], defaults=[]), body=[Assign(targets=[Name(id='id_1', ctx=Store())], value=Constant(value=0)), For(target=Name(id='id_2', ctx=Store()), iter=Call(func=Name(id='id_3', ctx=Load()), args=[Call(func=Name(id='id_4', ctx=Load()), args=[Name(id='id_0', ctx=Load())], keywords=[])], keywords=[]), body=[For(target=Name(id='id_5', ctx=Store()), iter=Call(func=Name(id='id_3', ctx=Load()), args=[Name(id='id_2', ctx=Load()), Call(func=Name(id='id_4', ctx=Load()), args=[Name(id='id_0', ctx=Load())], keywords=[])], keywords=[]), body=[Assign(targets=[Name(id='id_6', ctx=Store())], value=Subscript(value=Name(id='id_0', ctx=Load()), slice=Slice(lower=Name(id='id_2', ctx=Load()), upper=BinOp(left=Name(id='id_5', ctx=Load()), op=Add(), right=Constant(value=1))), ctx=Load())), Assign(targets=[Name(id='id_7', ctx=Store())], value=Call(func=Name(id='id_8', ctx=Load()), args=[GeneratorExp

In [44]:
normalize(base_code) == normalize(steered_code)

False

In [45]:
## remaining agenda.
### -- meta stats where base plans vs. where instruct plans and whether base was able to solbe.
### -- what category does the plan belong to -> category-1 (competing plans) OR category-2 (inducing new plans) OR category-3 (removal of existing plan)

In [46]:
og_pass_file = "../data/external/first_100_passing_examples_without_docstrings_base_model_og_prompt_V2.json"
og_fail_file = "../data/external/first_100_failing_examples_without_docstrings_base_model_og_prompt_V2.json"
og_pass = load_json(og_pass_file)
og_fail = load_json(og_fail_file)
og_pass_ids = [entry["task_id"] for entry in og_pass]
og_fail_ids = [entry["task_id"] for entry in og_fail]

In [47]:
selected_file = original_path
selected_data = load_json(selected_file)
selected_ids = [entry["task_id"] for entry in selected_data]

In [48]:
### setting whether base has passed OR failed here.
for entry in selected_data:
    if entry["task_id"] in og_pass_ids:
        entry["base_pass"] = True
    else:
        entry["base_pass"] = False

In [49]:
glob.glob("../outputs/t-scale-exp/base/prompt_0/*/*.json")

['../outputs/t-scale-exp/base/prompt_0/token_113/updated_planning_analysis.json',
 '../outputs/t-scale-exp/base/prompt_0/token_114/updated_planning_analysis.json',
 '../outputs/t-scale-exp/base/prompt_0/token_115/updated_planning_analysis.json',
 '../outputs/t-scale-exp/base/prompt_0/token_116/updated_planning_analysis.json',
 '../outputs/t-scale-exp/base/prompt_0/token_117/updated_planning_analysis.json']

In [50]:
def _detect_planning(planning_dict):
    keys = []
    for key, value in planning_dict.items():
        if value["new_verdict"] == "Plan":
            keys.append(key)
    return keys

def _get_ym_plans(folder):
    token_planning_files = glob.glob(folder + "/*/*.json")
    planning_datas = [load_json(f) for f in token_planning_files]
    y_ms = []
    for data in planning_datas:
        y_ms.extend(_detect_planning(data))
    y_ms = list(set(y_ms))
    return y_ms


def _get_base_and_instruct_plans(iter):
    base_folder = f"../outputs/t-scale-exp/base/prompt_{iter}"
    instruct_folder = f"../outputs/t-scale-exp/instruct/prompt_{iter}"
    if os.path.exists(base_folder) and os.path.exists(instruct_folder):
        base_yms = _get_ym_plans(base_folder)
        instruct_yms = _get_ym_plans(instruct_folder)
    else:
        base_yms = None
        instruct_yms = None
    return {
        "base": base_yms,
        "instruct": instruct_yms
    }

In [51]:
all_prompts = range(len(selected_ids))
len(all_prompts)

81

In [52]:
for iter, entry in enumerate(selected_data):
    ym_plans = _get_base_and_instruct_plans(iter)
    entry["base_plans"] = ym_plans["base"]
    entry["instruct_plans"] = ym_plans["instruct"]

In [53]:
# --- TABLE 1: ALL CASES ---
# Rows: instruct plans / does not plan
# Columns: base pass / base fail
table1 = {
    True:  {True: 0, False: 0},  # instruct plans
    False: {True: 0, False: 0}   # instruct does not plan
}

# --- TABLE 2: ONLY WHEN INSTRUCT PLANS ---
# Rows: base plans / base does not plan
# Columns: base pass / base fail
table2 = {
    True:  {True: 0, False: 0},  # base plans
    False: {True: 0, False: 0}   # base does not plan
}

# --- TABLE 3: ONLY WHEN INSTRUCT DOES *NOT* PLAN ---
# Rows: base plans / base does not plan
# Columns: base pass / base fail
table3 = {
    True:  {True: 0, False: 0},  # base plans
    False: {True: 0, False: 0}   # base does not plan
}

# --- TABLE 4: CONFUSION MATRIX OF INSTRUCT PLANS vs BASE PLANS ---
# Rows: instruct plans / instruct does not plan
# Columns: base plans / base does not plan
table4 = {
    True:  {True: 0, False: 0},  # instruct plans
    False: {True: 0, False: 0}   # instruct does not plan
}

for entry in selected_data:

    base_pass = bool(entry.get("base_pass"))
    base_plans_empty = isinstance(entry.get("base_plans"), list) and len(entry["base_plans"]) == 0
    instruct_plans_empty = isinstance(entry.get("instruct_plans"), list) and len(entry["instruct_plans"]) == 0

    instruct_plans = not instruct_plans_empty
    base_plans = not base_plans_empty

    # Table 1
    table1[instruct_plans][base_pass] += 1

    # Table 2
    if instruct_plans:
        table2[base_plans][base_pass] += 1

    # Table 3
    else:
        table3[base_plans][base_pass] += 1

    # --- Table 4: confusion matrix (instruct plans vs base plans) ---
    table4[instruct_plans][base_plans] += 1


# Pretty print

print("\n=== TABLE 1: ALL CASES ===")
print("                    Base Pass | Base Fail")
print("------------------------------------------")
print(f"Instruct Plans       {table1[True][True]:9d} | {table1[True][False]:9d}")
print(f"Instruct No-Plan     {table1[False][True]:9d} | {table1[False][False]:9d}")

print("\n=== TABLE 2: ONLY CASES WHERE INSTRUCT PLANS ===")
print("                     Base Pass | Base Fail")
print("-------------------------------------------")
print(f"Base Plans           {table2[True][True]:9d} | {table2[True][False]:9d}")
print(f"Base No-Plan         {table2[False][True]:9d} | {table2[False][False]:9d}")

print("\n=== TABLE 3: ONLY CASES WHERE INSTRUCT DOES NOT PLAN ===")
print("                     Base Pass | Base Fail")
print("-------------------------------------------")
print(f"Base Plans           {table3[True][True]:9d} | {table3[True][False]:9d}")
print(f"Base No-Plan         {table3[False][True]:9d} | {table3[False][False]:9d}")

print("\n=== TABLE 4: CONFUSION MATRIX (INSTRUCT PLANS vs BASE PLANS) ===")
print("                     Base Plans | Base No-Plan")
print("------------------------------------------------")
print(f"Instruct Plans       {table4[True][True]:11d} | {table4[True][False]:11d}")
print(f"Instruct No-Plan     {table4[False][True]:11d} | {table4[False][False]:11d}")


=== TABLE 1: ALL CASES ===
                    Base Pass | Base Fail
------------------------------------------
Instruct Plans              25 |        19
Instruct No-Plan            28 |         9

=== TABLE 2: ONLY CASES WHERE INSTRUCT PLANS ===
                     Base Pass | Base Fail
-------------------------------------------
Base Plans                  22 |        13
Base No-Plan                 3 |         6

=== TABLE 3: ONLY CASES WHERE INSTRUCT DOES NOT PLAN ===
                     Base Pass | Base Fail
-------------------------------------------
Base Plans                  12 |         6
Base No-Plan                16 |         3

=== TABLE 4: CONFUSION MATRIX (INSTRUCT PLANS vs BASE PLANS) ===
                     Base Plans | Base No-Plan
------------------------------------------------
Instruct Plans                35 |           9
Instruct No-Plan              18 |          19


In [54]:
sampled_cases = [(idx, entry) for idx, entry in enumerate(selected_data) if entry["base_pass"] == False and entry["instruct_plans"] is not None and len(entry["instruct_plans"]) == 0]
sampled_cases[0][1], len(sampled_cases)

({'source_file': "Mike's Copy of Benchmark Questions Verification V2.ipynb",
  'task_id': 96,
  'prompt': 'Write a python function to find the number of divisors of a given integer.',
  'code': 'def divisor(n):\n  for i in range(n):\n    x = len([i for i in range(1,n+1) if not n % i])\n  return x',
  'test_imports': [],
  'test_list': ['assert divisor(15) == 4',
   'assert divisor(12) == 6',
   'assert divisor(9) == 3'],
  'model_output': 'def divisor(n):\n    if n == 1:\n        return 1\n    else:\n        return divisor(n-1) + 1',
  'instruct_code': 'def divisor(n):\n    count = 0\n    for i in range(1, int(n**0.5) + 1):\n        if n % i == 0:\n            count += 1\n            if i != n // i:\n                count += 1\n    return count\n',
  'position_info': {'base_char_pos': 343,
   'instruct_char_pos': 364,
   'base_token_pos': 92,
   'instruct_token_pos': 97,
   'base_token_id': 141,
   'instruct_token_id': 141},
  'base_pass': False,
  'base_plans': ['-'],
  'instruct_plan

In [55]:
sampled_cases[0]

(9,
 {'source_file': "Mike's Copy of Benchmark Questions Verification V2.ipynb",
  'task_id': 96,
  'prompt': 'Write a python function to find the number of divisors of a given integer.',
  'code': 'def divisor(n):\n  for i in range(n):\n    x = len([i for i in range(1,n+1) if not n % i])\n  return x',
  'test_imports': [],
  'test_list': ['assert divisor(15) == 4',
   'assert divisor(12) == 6',
   'assert divisor(9) == 3'],
  'model_output': 'def divisor(n):\n    if n == 1:\n        return 1\n    else:\n        return divisor(n-1) + 1',
  'instruct_code': 'def divisor(n):\n    count = 0\n    for i in range(1, int(n**0.5) + 1):\n        if n % i == 0:\n            count += 1\n            if i != n // i:\n                count += 1\n    return count\n',
  'position_info': {'base_char_pos': 343,
   'instruct_char_pos': 364,
   'base_token_pos': 92,
   'instruct_token_pos': 97,
   'base_token_id': 141,
   'instruct_token_id': 141},
  'base_pass': False,
  'base_plans': ['-'],
  'instruct_

In [56]:
sampled_cases[1]

(10,
 {'source_file': "Mike's Copy of Benchmark Questions Verification V2.ipynb",
  'task_id': 129,
  'prompt': 'Write a function to calculate whether the matrix is a magic square.',
  'code': 'def magic_square_test(my_matrix):\n    iSize = len(my_matrix[0])\n    sum_list = []\n    sum_list.extend([sum (lines) for lines in my_matrix])   \n    for col in range(iSize):\n        sum_list.append(sum(row[col] for row in my_matrix))\n    result1 = 0\n    for i in range(0,iSize):\n        result1 +=my_matrix[i][i]\n    sum_list.append(result1)      \n    result2 = 0\n    for i in range(iSize-1,-1,-1):\n        result2 +=my_matrix[i][i]\n    sum_list.append(result2)\n    if len(set(sum_list))>1:\n        return False\n    return True',
  'test_imports': [],
  'test_list': ['assert magic_square_test([[7, 12, 1, 14], [2, 13, 8, 11], [16, 3, 10, 5], [9, 6, 15, 4]])==True',
   'assert magic_square_test([[2, 7, 6], [9, 5, 1], [4, 3, 8]])==True',
   'assert magic_square_test([[2, 7, 6], [9, 5, 1], [

In [57]:
sampled_cases[2]

(11,
 {'source_file': "Mike's Copy of Benchmark Questions Verification V2.ipynb",
  'task_id': 132,
  'prompt': 'Write a function to convert a tuple to a string.',
  'code': "def tup_string(tup1):\n  str =  ''.join(tup1)\n  return str",
  'test_imports': [],
  'test_list': ['assert tup_string((\'e\', \'x\', \'e\', \'r\', \'c\', \'i\', \'s\', \'e\', \'s\'))==("exercises")',
   'assert tup_string((\'p\',\'y\',\'t\',\'h\',\'o\',\'n\'))==("python")',
   'assert tup_string((\'p\',\'r\',\'o\',\'g\',\'r\',\'a\',\'m\'))==("program")'],
  'model_output': 'def tup_string(tup):\n    return str(tup)',
  'instruct_code': "def tup_string(tup):\n    return ''.join(tup)\n",
  'position_info': {'base_char_pos': 457,
   'instruct_char_pos': 478,
   'base_token_pos': 144,
   'instruct_token_pos': 149,
   'base_token_id': 773,
   'instruct_token_id': 773},
  'base_pass': False,
  'base_plans': [],
  'instruct_plans': []})

In [58]:
sampled_cases[3]

(13,
 {'source_file': 'Benchmark Questions Verification V2.ipynb',
  'task_id': 162,
  'prompt': 'Write a function to calculate the sum (n - 2*i) from i=0 to n // 2, for instance n + (n-2) + (n-4)... (until n-x =< 0).',
  'code': 'def sum_series(n):\n  if n < 1:\n    return 0\n  else:\n    return n + sum_series(n - 2)',
  'test_imports': [],
  'test_list': ['assert sum_series(6) == 12',
   'assert sum_series(10) == 30',
   'assert sum_series(9) == 25'],
  'model_output': 'def sum_series(n):\n    return n - 2 * (n // 2)',
  'instruct_code': 'def sum_series(n):\n  total = 0\n  for i in range(n // 2):\n    total += n - 2 * i\n  return total\n',
  'position_info': {'base_char_pos': 400,
   'instruct_char_pos': 421,
   'base_token_pos': 136,
   'instruct_token_pos': 142,
   'base_token_id': 108,
   'instruct_token_id': 139},
  'base_pass': False,
  'base_plans': [],
  'instruct_plans': []})

In [59]:
sampled_cases[4]

(16,
 {'source_file': 'Benchmark Questions Verification V2.ipynb',
  'task_id': 170,
  'prompt': 'Write a function to find the sum of numbers in a list within a range specified by two indices.',
  'code': 'def sum_range_list(list1, m, n):                                                                                                                                                                                                \n    sum_range = 0                                                                                                                                                                                                         \n    for i in range(m, n+1, 1):                                                                                                                                                                                        \n        sum_range += list1[i]                                                                                                       

In [60]:
for task in sampled_cases:
    print(task[0])

9
10
11
13
16
17
19
23
26


In [61]:
def sum_range_list(list, start, end):
    return sum(list[start:end])

In [62]:
sum_range_list([2,1,5,6,8,3,4,9,10,11,8,12], 8, 10)

21

In [83]:
# data/external/first_100_selected_examples_without_docstrings_base_model_og_prompt_V2_instruct_comparison_with_position_info.json

import json

with open(
    "../data/external/first_100_selected_examples_without_docstrings_base_model_og_prompt_V2_instruct_comparison_with_position_info_post_process.json",
    "w"
) as f:
    json.dump(selected_data, f, indent=2)

In [87]:
def _check_if_codes_are_similar(base_code, instruct_code, threshold = 0.95):
    normalized_base_code = normalize(base_code)
    normalized_instruct_code = normalize(instruct_code)
    if normalized_base_code != None and normalized_instruct_code != None:
        return difflib.SequenceMatcher(None, normalized_base_code, normalized_instruct_code).ratio() > threshold
    return difflib.SequenceMatcher(None, base_code, instruct_code).ratio() > (threshold - 0.05)

def _does_base_answer_steer_to_instruct(iter, valid_yms, instruct_code, threshold = 0.95):
    """
        Classify whether the base answer ever steer towards instruct OR not.

        Returns:
            all_steers: the y_ms whose steers are close to the actual answer.
    """
    all_steers = []
    base_folder = f"../outputs/scale-exp/base/prompt_{iter}/*/"
    all_token_folders = glob.glob(base_folder)
    for folder in all_token_folders:
        steering_results = folder + "steering_results.json"
        metadata_json = folder + "metadata.json"
        token_idx = folder.split("token_")[1].split("/")[0]
        with open(metadata_json, "r") as f:
            base_input_prefix = json.load(f)["input_prefix_text"]
        with open(steering_results, "r") as f:
            steering_data = json.load(f)
        for key, value in steering_data.items():
            if key in valid_yms:
                for coeff_entry in value["steered"]:
                    if "decoded_text" in coeff_entry:
                        base_code = _extract_code_part(base_input_prefix, coeff_entry["decoded_text"])
                        if _check_if_codes_are_similar(base_code, instruct_code, threshold):
                            all_steers.append((token_idx, key))

    all_steers = list(set(all_steers))                
    return all_steers

In [88]:
def get_category_wise_diff():
    file = "../data/external/first_100_selected_examples_without_docstrings_base_model_og_prompt_V2_instruct_comparison_with_position_info_post_process.json"
    with open(file, "r") as f:
        post_process_data = json.load(f)
    for iter, entry in enumerate(post_process_data):
        class_case = None
        steers = []
        base_len = len(entry["base_plans"]) if entry["base_plans"] is not None else 0
        instruct_len = len(entry["instruct_plans"]) if entry["instruct_plans"] is not None else 0
        if base_len == 0 and instruct_len == 0:
            class_case = 4 ## both instruct and base don't plan.
        elif instruct_len == 0:
            class_case = 3 ## only base plans.
        else:
            steers =  _does_base_answer_steer_to_instruct(iter, entry["base_plans"], entry["instruct_code"])
            class_case = 1 if len(steers) > 0 else 2 ## 1 means the instruct answer is still present in base for a given steer AND 2 means it's not reliably present here.
        entry["category"] = class_case
        entry["steers"] = steers
    with open(file, "w") as f:
        json.dump(post_process_data, f, indent = 2)

In [89]:
get_category_wise_diff()

In [90]:
file = "../data/external/first_100_selected_examples_without_docstrings_base_model_og_prompt_V2_instruct_comparison_with_position_info_post_process.json"
with open(file, "r") as f:
    post_process_data = json.load(f)
post_process_data[0]

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 3,
 'prompt': 'Write a python function to identify non-prime numbers.',
 'code': 'import math\ndef is_not_prime(n):\n    result = False\n    for i in range(2,int(math.sqrt(n)) + 1):\n        if n % i == 0:\n            result = True\n    return result',
 'test_imports': [],
 'test_list': ['assert is_not_prime(2) == False',
  'assert is_not_prime(10) == True',
  'assert is_not_prime(35) == True',
  'assert is_not_prime(37) == False'],
 'model_output': 'def is_not_prime(number):\n    if number == 2:\n        return False\n    if number == 1:\n        return True\n    if number % 2 == 0:\n        return False\n    for i in range(3, number, 2):\n        if number % i == 0:\n            return False\n    return True',
 'instruct_code': 'def is_not_prime(n):\n    if n <= 1:\n        return True\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return True\n    return False\n',
 'position_info

In [93]:
from collections import defaultdict

def category_vs_base_pass(records):
    # table[category][base_pass] = count
    table = defaultdict(lambda: {True: 0, False: 0})

    for r in records:
        cat = r["category"]
        bp = r["base_pass"]
        table[cat][bp] += 1

    # pretty print
    print("Category | Base Pass | Base Fail | Meaning")
    print("---------------------------------------------------------------")
    meanings = {
        1: "Instruct plans ∧ Base has those plans",
        2: "Instruct plans ∧ Base lacks those plans",
        3: "Base plans ∧ Instruct does not",
        4: "Neither plans",
    }
    for cat in [1,2,3,4]:
        bp = table[cat]
        print(f"{cat:8} | {bp[True]:9} | {bp[False]:9} | {meanings[cat]}")

category_vs_base_pass(post_process_data)

Category | Base Pass | Base Fail | Meaning
---------------------------------------------------------------
       1 |         8 |         3 | Instruct plans ∧ Base has those plans
       2 |        17 |        15 | Instruct plans ∧ Base lacks those plans
       3 |        12 |         6 | Base plans ∧ Instruct does not
       4 |        16 |         4 | Neither plans


In [105]:
categories = [[],[],[],[],[]]
for iter, entry in enumerate(post_process_data):
    if entry["base_pass"] == False:
        categories[entry["category"]].append(iter)

In [106]:
categories[1]

[2, 15, 27]

In [109]:
post_process_data[2]

{'source_file': "Mike's Copy of Benchmark Questions Verification V2.ipynb",
 'task_id': 56,
 'prompt': 'Write a python function to check if a given number is one less than twice its reverse.',
 'code': 'def rev(num):    \n    rev_num = 0\n    while (num > 0):  \n        rev_num = (rev_num * 10 + num % 10) \n        num = num // 10  \n    return rev_num  \ndef check(n):    \n    return (2 * rev(n) == n + 1)  ',
 'test_imports': [],
 'test_list': ['assert check(70) == False',
  'assert check(23) == False',
  'assert check(73) == True'],
 'model_output': 'def check(n):\n    if n == 0:\n        return False\n    if n < 0:\n        n = -n\n    rev = 0\n    while n > 0:\n        rev = rev * 10 + n % 10\n        n = n // 10\n    if n == 0:\n        return True\n    if n == rev - 1:\n        return True\n    return False',
 'instruct_code': 'def check(n):\n  return n == 2 * int(str(n)[::-1]) - 1\n',
 'position_info': {'base_char_pos': 357,
  'instruct_char_pos': 378,
  'base_token_pos': 92,
  

In [110]:
post_process_data[15]

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 166,
 'prompt': 'Write a function that counts the number of pairs of integers in a list that xor to an even number.',
 'code': 'def find_even_pair(A): \n  count = 0\n  for i in range(0, len(A)): \n    for j in range(i+1, len(A)): \n        if ((A[i] ^ A[j]) % 2 == 0): \n          count += 1\n\n  return count',
 'test_imports': [],
 'test_list': ['assert find_even_pair([5, 4, 7, 2, 1]) == 4',
  'assert find_even_pair([7, 2, 8, 1, 0, 5, 11]) == 9',
  'assert find_even_pair([1, 2, 3]) == 1'],
 'model_output': 'def find_even_pair(list):\n    even_pairs = 0\n    for i in range(len(list)):\n        for j in range(i+1, len(list)):\n            if list[i] ^ list[j] == 0:\n                even_pairs += 1\n    return even_pairs',
 'instruct_code': 'def find_even_pair(nums):\n  count = 0\n  for i in range(len(nums)):\n    for j in range(i + 1, len(nums)):\n      if (nums[i] ^ nums[j]) % 2 == 0:\n        count += 1\n  return 

In [111]:
post_process_data[27]

{'source_file': "Ellen's Copy of Benchmark Questions Verification V2.ipynb",
 'task_id': 312,
 'prompt': 'Write a function to find the volume of a cone.',
 'code': 'import math\ndef volume_cone(r,h):\n  volume = (1.0/3) * math.pi * r * r * h\n  return volume',
 'test_imports': ['import math'],
 'test_list': ['assert math.isclose(volume_cone(5,12), 314.15926535897927, rel_tol=0.001)',
  'assert math.isclose(volume_cone(10,15), 1570.7963267948965, rel_tol=0.001)',
  'assert math.isclose(volume_cone(19,17), 6426.651371693521, rel_tol=0.001)'],
 'model_output': 'def volume_cone(radius, height):\n    return (1/3) * math.pi * radius ** 2 * height',
 'instruct_code': 'import math\n\ndef volume_cone(radius, height):\n    return (1/3) * math.pi * radius**2 * height\n',
 'position_info': {'base_char_pos': 515,
  'instruct_char_pos': 549,
  'base_token_pos': 214,
  'instruct_token_pos': 222,
  'base_token_id': 16316,
  'instruct_token_id': 16316},
 'base_pass': False,
 'base_plans': ['**'],
 'ins

In [107]:
categories[2]

[0, 1, 3, 4, 5, 6, 7, 8, 14, 18, 20, 21, 22, 24, 25]

In [112]:
post_process_data[3]

{'source_file': "Mike's Copy of Benchmark Questions Verification V2.ipynb",
 'task_id': 61,
 'prompt': 'Write a python function to count the number of substrings with the sum of digits equal to their length.',
 'code': "from collections import defaultdict\ndef count_Substrings(s):\n    n = len(s)\n    count,sum = 0,0\n    mp = defaultdict(lambda : 0)\n    mp[0] += 1\n    for i in range(n):\n        sum += ord(s[i]) - ord('0')\n        count += mp[sum - (i + 1)]\n        mp[sum - (i + 1)] += 1\n    return count",
 'test_imports': [],
 'test_list': ["assert count_Substrings('112112') == 6",
  "assert count_Substrings('111') == 6",
  "assert count_Substrings('1101112') == 12"],
 'model_output': 'def count_Substrings(s):\n    return len(s) - len(set(s))',
 'instruct_code': 'def count_Substrings(s):\n    count = 0\n    for i in range(len(s)):\n        for j in range(i, len(s)):\n            substring = s[i:j+1]\n            sum_digits = sum(int(digit) for digit in substring)\n            if

In [113]:
post_process_data[5]

{'source_file': "Mike's Copy of Benchmark Questions Verification V2.ipynb",
 'task_id': 69,
 'prompt': 'Write a function to check whether a list contains the given sublist or not.',
 'code': 'def is_sublist(l, s):\n\tsub_set = False\n\tif s == []:\n\t\tsub_set = True\n\telif s == l:\n\t\tsub_set = True\n\telif len(s) > len(l):\n\t\tsub_set = False\n\telse:\n\t\tfor i in range(len(l)):\n\t\t\tif l[i] == s[0]:\n\t\t\t\tn = 1\n\t\t\t\twhile (n < len(s)) and (l[i+n] == s[n]):\n\t\t\t\t\tn += 1\t\t\t\t\n\t\t\t\tif n == len(s):\n\t\t\t\t\tsub_set = True\n\treturn sub_set',
 'test_imports': [],
 'test_list': ['assert is_sublist([2,4,3,5,7],[3,7])==False',
  'assert is_sublist([2,4,3,5,7],[4,3])==True',
  'assert is_sublist([2,4,3,5,7],[1,6])==False'],
 'model_output': 'def is_sublist(list1, list2):\n    if len(list1) < len(list2):\n        return False\n    for i in range(len(list1)):\n        if list1[i] in list2:\n            return True\n    return False',
 'instruct_code': 'def is_sublist

In [108]:
categories[3]

[9, 16, 17, 19, 23, 26]

In [114]:
post_process_data[9]

{'source_file': "Mike's Copy of Benchmark Questions Verification V2.ipynb",
 'task_id': 96,
 'prompt': 'Write a python function to find the number of divisors of a given integer.',
 'code': 'def divisor(n):\n  for i in range(n):\n    x = len([i for i in range(1,n+1) if not n % i])\n  return x',
 'test_imports': [],
 'test_list': ['assert divisor(15) == 4',
  'assert divisor(12) == 6',
  'assert divisor(9) == 3'],
 'model_output': 'def divisor(n):\n    if n == 1:\n        return 1\n    else:\n        return divisor(n-1) + 1',
 'instruct_code': 'def divisor(n):\n    count = 0\n    for i in range(1, int(n**0.5) + 1):\n        if n % i == 0:\n            count += 1\n            if i != n // i:\n                count += 1\n    return count\n',
 'position_info': {'base_char_pos': 343,
  'instruct_char_pos': 364,
  'base_token_pos': 92,
  'instruct_token_pos': 97,
  'base_token_id': 141,
  'instruct_token_id': 141},
 'base_pass': False,
 'base_plans': ['-'],
 'instruct_plans': [],
 'category'

In [115]:
post_process_data[16]

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 170,
 'prompt': 'Write a function to find the sum of numbers in a list within a range specified by two indices.',
 'code': 'def sum_range_list(list1, m, n):                                                                                                                                                                                                \n    sum_range = 0                                                                                                                                                                                                         \n    for i in range(m, n+1, 1):                                                                                                                                                                                        \n        sum_range += list1[i]                                                                                                                

In [117]:
post_process_data[17]

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 224,
 'prompt': 'Write a python function to count the number of set bits (binary digits with value 1) in a given number.',
 'code': 'def count_Set_Bits(n): \n    count = 0\n    while (n): \n        count += n & 1\n        n >>= 1\n    return count ',
 'test_imports': [],
 'test_list': ['assert count_Set_Bits(2) == 1',
  'assert count_Set_Bits(4) == 1',
  'assert count_Set_Bits(6) == 2'],
 'model_output': 'def count_Set_Bits(n):\n    return sum(1 for i in range(1, n + 1) if i & n)',
 'instruct_code': 'def count_Set_Bits(n):\n    count = 0\n    while n:\n        count += n & 1\n        n >>= 1\n    return count\n',
 'position_info': {'base_char_pos': 398,
  'instruct_char_pos': 419,
  'base_token_pos': 115,
  'instruct_token_pos': 120,
  'base_token_id': 141,
  'instruct_token_id': 141},
 'base_pass': False,
 'base_plans': ['sum', 'i', 'for', '+'],
 'instruct_plans': [],
 'category': 3,
 'steers': []}